# 🏦 Intelligent Multi-Agent Credit Committee & Fraud Detection
### کمیته‌ی هوشمند و چندعاملی تخصیص اعتبار و کشف تقلب

**Generative AI Bootcamp — Final Project**

---

## 📌 Project Goal
In a real bank, a loan request is not approved by a single person. It goes to a **credit committee**:
one person checks the customer's identity, another checks bounced cheques and debts, a legal expert
checks the latest Central Bank regulations, and finally a manager makes the decision.

In this project we **simulate that committee with AI agents** that talk to each other inside a
**LangGraph** decision graph. Together they review the request and calculate the customer's risk score.

## 🧩 System Architecture (the 4 agents)

| # | Agent (Node) | Role | Main job | Output |
|---|---|---|---|---|
| 1 | **Triage & Sentiment Agent** | Front desk | Welcomes the customer, checks the documents, detects the tone (sentiment) | `Angry` ➜ stop & refer to a human operator · `Normal` ➜ continue |
| 2 | **Risk & Fraud Agent** *(Tool Caller)* | Financial risk analyst | Calls mock banking APIs with the national ID (bounced cheques, current debts, blacklist) | Credit score + red flags |
| 3 | **Compliance Agent** *(RAG)* | Legal / regulations expert | Searches the Central Bank circulars stored in a Vectorstore | Is the request legal? Which conditions apply? |
| 4 | **Supervisor / Judge** | Head of the credit committee | Combines reports of agents 2 and 3 | Final structured **JSON**: status, max amount, collateral |

## 🔀 Workflow
```
              ┌──────────────────────┐
  Customer ─► │ 1. Triage & Sentiment│
              └──────────┬───────────┘
          Angry ◄────────┴────────► Normal
            │                    ┌──────┴───────┐
            ▼                    ▼              ▼
     Human operator      2. Risk & Fraud   3. Compliance
          (END)              (Tools)          (RAG)
                                 └──────┬───────┘
                                        ▼
                              4. Supervisor / Judge ─► JSON ─► END
```

## 🛠️ Tech Stack
| Component | Tool |
|---|---|
| Agent orchestration | **LangGraph** |
| LLM (runs locally) | **Ollama** – `qwen2.5:7b` |
| Embeddings for RAG | **Ollama** – `bge-m3` (multilingual) |
| Vectorstore | **Chroma** |
| Tools | LangChain `@tool` decorator (mock banking APIs) |

## 🗺️ Roadmap of this notebook
- [x] **Step 0** – Environment setup & connecting to Ollama
- [x] **Step 1** – Defining the State (`BankingState`)
- [x] **Step 2** – Building the banking tools (Tool Calling)
- [x] **Step 3** – Building the agents (nodes) and conditional edges
  - [x] 3.1 Triage & Sentiment Agent (welcome, sentiment, collecting the documents)
  - [x] 3.2 Risk & Fraud Agent (Tool Calling)
  - [x] 3.3 Compliance Agent (RAG + Vectorstore)
  - [x] 3.4 Supervisor / Judge (structured JSON)
  - [x] 3.5 Connecting the full graph (conditional edges)
- [x] **Step 4** – Test scenarios & graph visualization

---
# ⚙️ Step 0 — Environment Setup

Before building the agents, we prepare the environment:
1. Install the required libraries.
2. Import the basic modules.
3. Make sure **Ollama** is running and our models are available, and send a first test message to the LLM.

> 💡 **Why Ollama?** The model runs **locally** on our own machine. No customer data (national ID,
> loan amount, …) leaves the computer — which is very important for a banking application.

### 0.1 Install the libraries
| Library | Why we need it |
|---|---|
| `langgraph` | Builds the multi-agent graph (nodes, edges, state) |
| `langchain`, `langchain-core` | Base classes: messages, prompts, `@tool` decorator |
| `langchain-ollama` | Connects LangChain to the local Ollama models (`ChatOllama`, `OllamaEmbeddings`) |
| `langchain-community`, `pypdf` | Document loaders for the circulars (text / PDF) |
| `langchain-text-splitters` | Cuts the circulars into smaller chunks for RAG |
| `langchain-chroma`, `chromadb` | The Vectorstore for the RAG agent |

*(Run this cell only once. Restart the kernel afterwards if a package was newly installed.)*

In [30]:
# Install all the packages we need for the whole project (only needed once)
%pip install -q langgraph langchain langchain-core langchain-ollama langchain-community langchain-text-splitters langchain-chroma chromadb pypdf

Note: you may need to restart the kernel to use updated packages.


### 0.2 Import the basic modules
We import only what we need for now. Each next step will import its own extra modules, so it is
always clear **which library is used for what**.

In [31]:
# ---- Standard Python libraries ----
import json                                   # to pretty-print dictionaries (State, reports, final JSON)
import operator                               # operator.add -> used as a "reducer" in the State
from typing import TypedDict, Annotated       # to define the structure (schema) of our State

# ---- LangChain / LangGraph ----
from langchain_ollama import ChatOllama       # chat model served by Ollama
from langgraph.graph import StateGraph, START, END   # graph builder + special start/end nodes


In [32]:
# ---------------- Model configuration (one central place) ----------------
OLLAMA_BASE_URL = "http://localhost:11434"   # default address of the local Ollama server
LLM_MODEL       = "qwen2.5:7b"               # chat model used by all agents
EMBEDDING_MODEL = "bge-m3"                   # multilingual embedding model (good for Persian), used for RAG

In [33]:
# ---------------- 3) Create the LLM object and send a first test message ----------------
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,   # deterministic answers -> consistent banking decisions
)

test_answer = llm.invoke("In one short sentence, introduce yourself as a bank assistant.")
print("🤖 LLM test answer:", test_answer.content)

🤖 LLM test answer: I'm Qwen, here to assist you with all your banking needs and inquiries.


In [34]:
# =====================================================================
#  STEP 1: The State of the credit committee graph
# =====================================================================
class BankingState(TypedDict):
    # ---------------- Required fields (from the project brief) ----------------
    customer_id: str          # Customer ID or national ID (کد ملی)
    loan_request: str         # Requested loan amount and type
    sentiment: str            # Customer's emotional state (e.g. 'Normal' or 'Angry')
    risk_report: dict         # Output of the financial inquiries (risk / credit score)
    compliance_check: str     # Result of checking the circulars via RAG
    final_decision: dict      # Final structured output (JSON)

    # ---------------- Helper fields (our additions) ----------------
    user_input: str           # The customer's original raw message
    documents_ok: bool        # Initial document check by Triage (national ID + loan info present?)
    route: str                # Triage routing decision: 'credit_committee' or 'human_operator'
    messages: Annotated[list, operator.add]   # Conversation log; the reducer APPENDS new messages


In [35]:
def create_initial_state(user_input: str) -> BankingState:
    # Opens a new, empty case file for a customer message
    return BankingState(
        customer_id="",          # will be extracted by the Triage agent
        loan_request="",         # will be extracted by the Triage agent
        sentiment="",            # will be detected by the Triage agent
        risk_report={},          # will be filled by the Risk & Fraud agent
        compliance_check="",     # will be filled by the Compliance agent
        final_decision={},       # will be filled by the Supervisor
        user_input=user_input,   # the only field known at the start
        documents_ok=False,      # not checked yet
        route="",                # not decided yet
        messages=[],             # conversation log starts empty
    )

## Helper function 2: printing the State
**print the State values at each step** 
We use `json.dumps(..., ensure_ascii=False)` so that **Persian text is shown correctly**
(and not as `\u06a9\u062f...` codes).

In [36]:
def print_state(state: dict, title: str = "Current State") -> None:
    # Prints the case file (State) in a readable format
    print("=" * 70)
    print(f"📁 {title}")
    print("=" * 70)
    for key, value in state.items():
        if key == "messages":
            # print the conversation log one message per line
            print(f"• {key}:")
            for msg in value:
                print(f"    [{msg['agent']}] {msg['content']}")
        elif isinstance(value, dict) and value:
            # pretty-print non-empty dictionaries (risk_report, final_decision)
            pretty = json.dumps(value, ensure_ascii=False, indent=4).replace("\n", "\n    ")
            print(f"• {key}:\n    {pretty}")
        elif isinstance(value, str):
            # strings are shown inside quotes so empty fields ('') are visible
            print(f"• {key}: '{value}'")
        else:
            print(f"• {key}: {value}")
    print("=" * 70)

In [37]:
# The customer message of Scenario 1 (from the project brief)
scenario_1_input = "من برای توسعه کسب‌وکارم ۵۰۰ میلیون تومان وام خرید تجهیزات میخوام. کد ملی: ۱۲۳۴۵۶۷۸۹۰"

state = create_initial_state(scenario_1_input)
print_state(state, "Initial State — Scenario 1 (before any agent runs)")

📁 Initial State — Scenario 1 (before any agent runs)
• customer_id: ''
• loan_request: ''
• sentiment: ''
• risk_report: {}
• compliance_check: ''
• final_decision: {}
• user_input: 'من برای توسعه کسب‌وکارم ۵۰۰ میلیون تومان وام خرید تجهیزات میخوام. کد ملی: ۱۲۳۴۵۶۷۸۹۰'
• documents_ok: False
• route: ''
• messages:


### 2.1 Import the tool modules

In [38]:
from langchain_core.tools import tool                                       # decorator that turns a function into an LLM tool
from langchain_core.messages import SystemMessage, HumanMessage              # message types used to talk to the LLM


### 2.2 The mock bank database


| National ID | Profile | Used for |
|---|---|---|
| `1234567890` | ✅ Good customer: score 800, no bounced cheques, no overdue debt | **Scenario 1** of the brief |
| `0987654321` | ⚠️ Risky customer: low score, bounced cheques, high debt | Showing red flags |
| `1111111111` | ⛔ Blacklisted customer (cheque fraud) | Showing fraud detection |

Any other national ID is treated as a customer with **no record** in the banking network.
All amounts are in **Toman**, same as the customer's request.

In [39]:
# ----------------------------------------------------------------------
#  MOCK BANK DATABASE  (simulates the data behind the banking APIs)
#  key = national ID, value = the customer's records in different bank systems
#  All amounts are in Toman
# ----------------------------------------------------------------------
MOCK_BANK_DB = {
    # ✅ Scenario 1: excellent customer
    "1234567890": {
        "full_name": "Ali Rezaei",
        "credit_score": 800,
        "bounced_checks": 0,
        "bounced_checks_amount": 0,
        "active_loans": 1,
        "total_debt": 50_000_000,
        "overdue_installments": 0,
        "blacklisted": False,
        "blacklist_reason": "",
    },
    # ⚠️ Risky customer
    "0987654321": {
        "full_name": "Sara Ahmadi",
        "credit_score": 540,
        "bounced_checks": 3,
        "bounced_checks_amount": 120_000_000,
        "active_loans": 3,
        "total_debt": 900_000_000,
        "overdue_installments": 4,
        "blacklisted": False,
        "blacklist_reason": "",
    },
    # ⛔ Blacklisted customer
    "1111111111": {
        "full_name": "Reza Karimi",
        "credit_score": 320,
        "bounced_checks": 7,
        "bounced_checks_amount": 2_300_000_000,
        "active_loans": 2,
        "total_debt": 1_500_000_000,
        "overdue_installments": 9,
        "blacklisted": True,
        "blacklist_reason": "Cheque fraud case referred to the judiciary",
    },
}

print(f"✅ Mock bank database loaded with {len(MOCK_BANK_DB)} customers.")

✅ Mock bank database loaded with 3 customers.


In [40]:
############ cleaning the national ID #################


# Translation table: Persian digits (۰-۹) and Arabic digits (٠-٩) -> English digits (0-9)
DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def normalize_national_id(national_id: str) -> str:
    # Converts a national ID written in any format to a clean 10-digit English string.
    # Returns an empty string if the ID is not valid.
    cleaned = str(national_id).translate(DIGIT_MAP)            # Persian/Arabic digits -> English
    cleaned = "".join(ch for ch in cleaned if ch.isdigit())    # remove spaces, dashes, etc.
    return cleaned if len(cleaned) == 10 else ""               # a national ID has exactly 10 digits


# Quick test of the helper
for sample in ["۱۲۳۴۵۶۷۸۹۰", "١٢٣٤٥٦٧٨٩٠", "123-456-7890", "12345"]:
    result = normalize_national_id(sample)
    print(f"{sample:>14}  ->  {result if result else '❌ invalid'}")

    ۱۲۳۴۵۶۷۸۹۰  ->  1234567890
    ١٢٣٤٥٦٧٨٩٠  ->  1234567890
  123-456-7890  ->  1234567890
         12345  ->  ❌ invalid


### 2.4 Defining the tools with `@tool`
Risk & Fraud agent in Step 3

In [41]:
# ----------------------------------------------------------------------
#  Small internal helper used by all tools (not a tool itself)
# ----------------------------------------------------------------------
def _find_customer(national_id: str):
    # Returns (clean_id, record) or (clean_id, None) if the customer has no record
    clean_id = normalize_national_id(national_id)
    return clean_id, MOCK_BANK_DB.get(clean_id)


# ======================================================================
#  TOOL 1: Credit bureau
# ======================================================================
@tool
def get_credit_score(national_id: str) -> dict:

    """Get the customer's credit score (300-850) and credit rating from the credit bureau.
    Use this tool to evaluate how trustworthy the customer is for a loan.
    Input: the customer's 10-digit national ID."""

    clean_id, customer = _find_customer(national_id)
    if not clean_id:
        return {"source": "Credit Bureau", "error": "Invalid national ID (must be 10 digits)."}
    if customer is None:
        # no credit history -> a neutral starting score
        return {"source": "Credit Bureau", "national_id": clean_id, "credit_score": 600,
                "rating": "No credit history"}
    

    score = customer["credit_score"]
    # turn the numeric score into a human-readable rating
    if score >= 750:
        rating = "Excellent"
    elif score >= 650:
        rating = "Good"
    elif score >= 550:
        rating = "Fair"
    else:
        rating = "Poor"
    return {"source": "Credit Bureau", "national_id": clean_id, "credit_score": score, "rating": rating}


# ======================================================================
#  TOOL 2: Central Bank cheque system
# ======================================================================
@tool
def check_bounced_checks(national_id: str) -> dict:
    
    """Check the Central Bank cheque system for the customer's bounced (returned) cheques.
    Returns the number of bounced cheques and their total amount in Toman.
    Input: the customer's 10-digit national ID."""
    
    clean_id, customer = _find_customer(national_id)
    if not clean_id:
        return {"source": "Central Bank Cheque System", "error": "Invalid national ID (must be 10 digits)."}
    if customer is None:
        return {"source": "Central Bank Cheque System", "national_id": clean_id,
                "bounced_checks": 0, "bounced_checks_amount": 0}
    
    return {"source": "Central Bank Cheque System", "national_id": clean_id,
            "bounced_checks": customer["bounced_checks"],
            "bounced_checks_amount": customer["bounced_checks_amount"]}


# ======================================================================
#  TOOL 3: Banking network loan records
# ======================================================================
@tool
def check_current_debts(national_id: str) -> dict:
    """Check the customer's current debts in the banking network:
    number of active loans, total remaining debt in Toman, and number of overdue installments.
    Input: the customer's 10-digit national ID."""
    clean_id, customer = _find_customer(national_id)
    if not clean_id:
        return {"source": "Banking Network Loans", "error": "Invalid national ID (must be 10 digits)."}
    if customer is None:
        return {"source": "Banking Network Loans", "national_id": clean_id,
                "active_loans": 0, "total_debt": 0, "overdue_installments": 0}
    
    return {"source": "Banking Network Loans", "national_id": clean_id,
            "active_loans": customer["active_loans"],
            "total_debt": customer["total_debt"],
            "overdue_installments": customer["overdue_installments"]}


# ======================================================================
#  TOOL 4: Bank blacklist (fraud list)
# ======================================================================
@tool
def check_blacklist(national_id: str) -> dict:
    """Check whether the customer is on the bank's blacklist (fraud / banned customers list).
    Returns True/False and the reason if the customer is blacklisted.
    Input: the customer's 10-digit national ID."""
    clean_id, customer = _find_customer(national_id)
    if not clean_id:
        return {"source": "Bank Blacklist", "error": "Invalid national ID (must be 10 digits)."}
    if customer is None:
        return {"source": "Bank Blacklist", "national_id": clean_id, "blacklisted": False, "reason": ""}
    return {"source": "Bank Blacklist", "national_id": clean_id,
            "blacklisted": customer["blacklisted"], "reason": customer["blacklist_reason"]}


# All banking tools in one list -> this list is given to the Risk & Fraud agent in Step 3
banking_tools = [get_credit_score, check_bounced_checks, check_current_debts, check_blacklist]

# A name -> tool lookup table, used to run the tool that the LLM asks for
tools_by_name = {t.name: t for t in banking_tools}

print("✅ Banking tools created:", list(tools_by_name))

✅ Banking tools created: ['get_credit_score', 'check_bounced_checks', 'check_current_debts', 'check_blacklist']


### 2.5 What does the LLM "see" for each tool?
The `@tool` decorator created a **schema** for every function. This is exactly the information that is
sent to the LLM: name, description (from the docstring) and the input arguments.

In [42]:
# Show the name, description and input arguments of each tool (what the LLM reads)
for t in banking_tools:
    print(f"🔧 Tool name : {t.name}")
    print(f"   Arguments : {t.args}")
    print(f"   Description: {t.description}\n")

🔧 Tool name : get_credit_score
   Arguments : {'national_id': {'title': 'National Id', 'type': 'string'}}
   Description: Get the customer's credit score (300-850) and credit rating from the credit bureau.
    Use this tool to evaluate how trustworthy the customer is for a loan.
    Input: the customer's 10-digit national ID.

🔧 Tool name : check_bounced_checks
   Arguments : {'national_id': {'title': 'National Id', 'type': 'string'}}
   Description: Check the Central Bank cheque system for the customer's bounced (returned) cheques.
    Returns the number of bounced cheques and their total amount in Toman.
    Input: the customer's 10-digit national ID.

🔧 Tool name : check_current_debts
   Arguments : {'national_id': {'title': 'National Id', 'type': 'string'}}
   Description: Check the customer's current debts in the banking network:
    number of active loans, total remaining debt in Toman, and number of overdue installments.
    Input: the customer's 10-digit national ID.

🔧 Tool na

### 2.6 Test 1 — Calling the tools directly (without the LLM)
First we check that the tools return correct data. A tool is called with `.invoke({...})`.

We use the national ID of **Scenario 1**, written with **Persian digits** exactly like in the brief: `۱۲۳۴۵۶۷۸۹۰`.

**👀 What to look for:** credit score **800 (Excellent)**, **0** bounced cheques, not blacklisted (as the brief expects).

In [43]:
scenario_1_id = "۱۲۳۴۵۶۷۸۹۰"     # national ID of Scenario 1, with Persian digits

for banking_tool in banking_tools:
    result = banking_tool.invoke({"national_id": scenario_1_id})
    print(f"🔧 {banking_tool.name}:")
    print(f"   {result}")

🔧 get_credit_score:
   {'source': 'Credit Bureau', 'national_id': '1234567890', 'credit_score': 800, 'rating': 'Excellent'}
🔧 check_bounced_checks:
   {'source': 'Central Bank Cheque System', 'national_id': '1234567890', 'bounced_checks': 0, 'bounced_checks_amount': 0}
🔧 check_current_debts:
   {'source': 'Banking Network Loans', 'national_id': '1234567890', 'active_loans': 1, 'total_debt': 50000000, 'overdue_installments': 0}
🔧 check_blacklist:
   {'source': 'Bank Blacklist', 'national_id': '1234567890', 'blacklisted': False, 'reason': ''}


### 2.7 Test 2 — Comparing our 3 test customers
Now we ask about all 3 customers of our mock database, plus an **invalid** national ID.


In [44]:
for national_id in ["1234567890", "0987654321", "1111111111"]:
    score       = get_credit_score.invoke({"national_id": national_id})["credit_score"]
    bounced     = check_bounced_checks.invoke({"national_id": national_id})["bounced_checks"]
    blacklisted = check_blacklist.invoke({"national_id": national_id})["blacklisted"]
    print(f"National ID {national_id}: credit score = {score}, bounced cheques = {bounced}, blacklisted = {blacklisted}")

# An invalid national ID (only 5 digits)
print("\nInvalid ID test:", get_credit_score.invoke({"national_id": "12345"}))

National ID 1234567890: credit score = 800, bounced cheques = 0, blacklisted = False
National ID 0987654321: credit score = 540, bounced cheques = 3, blacklisted = False
National ID 1111111111: credit score = 320, bounced cheques = 7, blacklisted = True

Invalid ID test: {'source': 'Credit Bureau', 'error': 'Invalid national ID (must be 10 digits).'}


### 2.8 Test 3 — Can the LLM choose and call the tools by itself?
This is the real **Tool Calling** test:
1. `llm.bind_tools(banking_tools)` → we tell the LLM which tools exist.
2. We give it the **Persian message of Scenario 1**. We do **not** give it the national ID separately:
   it must **find it inside the sentence**.
3. `response.tool_calls` → the list of tools the LLM **decided** to call, with the arguments it chose.
4. Our code runs those tools.

**👀 What to look for:** the LLM asks for the tools with `national_id = '1234567890'` (found in the Persian text).

In [45]:
# 1) Give the LLM access to the tools
llm_with_tools = llm.bind_tools(banking_tools)

# 2) The role of the analyst + the customer's message (Scenario 1)
response = llm_with_tools.invoke([
    SystemMessage(content="You are a bank risk analyst. For the customer's national ID, call ALL the banking tools: "
                          "credit score, bounced cheques, current debts and blacklist."),
    HumanMessage(content=scenario_1_input),
])

# 3) Which tools did the LLM decide to call?
print(f"🤖 The LLM decided to call {len(response.tool_calls)} tools:")
for call in response.tool_calls:
    print(f"   🔧 {call['name']}  with arguments {call['args']}")

# 4) Our code runs the tools the LLM asked for
print("\n📋 Results:")
for call in response.tool_calls:
    tool_to_run = tools_by_name[call["name"]]        # find the tool by its name
    print(f"   {call['name']} -> {tool_to_run.invoke(call['args'])}")

🤖 The LLM decided to call 4 tools:
   🔧 check_blacklist  with arguments {'national_id': '1234567890'}
   🔧 get_credit_score  with arguments {'national_id': '1234567890'}
   🔧 check_bounced_checks  with arguments {'national_id': '1234567890'}
   🔧 check_current_debts  with arguments {'national_id': '1234567890'}

📋 Results:
   check_blacklist -> {'source': 'Bank Blacklist', 'national_id': '1234567890', 'blacklisted': False, 'reason': ''}
   get_credit_score -> {'source': 'Credit Bureau', 'national_id': '1234567890', 'credit_score': 800, 'rating': 'Excellent'}
   check_bounced_checks -> {'source': 'Central Bank Cheque System', 'national_id': '1234567890', 'bounced_checks': 0, 'bounced_checks_amount': 0}
   check_current_debts -> {'source': 'Banking Network Loans', 'national_id': '1234567890', 'active_loans': 1, 'total_debt': 50000000, 'overdue_installments': 0}


---
# 🤖 Step 3 — Building the Agents (Nodes)


### 3.1.1 Imports

In [46]:
import uuid                                          # unique thread_id (case number) for every conversation
from typing import Literal                           # restricts a field to fixed values, e.g. "Normal" / "Angry"
from pydantic import BaseModel, Field                # defines the structured output we want from the LLM
from langgraph.types import interrupt, Command       # pause the graph / resume it with the customer's answer
from langgraph.checkpoint.memory import InMemorySaver  # the graph's memory (needed for interrupt)

In [47]:
TRIAGE_AGENT = "Triage & Sentiment Agent"    
CUSTOMER     = "Customer"

# ---- Welcome ----
WELCOME_MESSAGE = (
    "سلام، وقت شما بخیر. به سامانه هوشمند «کمیته اعتباری» بانک خوش آمدید.\n"
    "من دستیار پذیرش هستم؛ درخواست شما را بررسی کرده و به کارشناس مربوطه ارجاع می‌دهم.\n"
    "لطفاً بفرمایید چه کمکی از دست من ساخته است؟"
)

# ---- Asking for missing documents (national ID first, then the loan) ----
ASK_NATIONAL_ID = "برای بررسی پرونده، لطفاً کد ملی ۱۰ رقمی خود را وارد بفرمایید."


INVALID_NATIONAL_ID = "کد ملی وارد شده معتبر نیست؛ کد ملی باید دقیقاً ۱۰ رقم باشد."


ASK_LOAN = ("سپاس‌گزارم. لطفاً نوع وام درخواستی (مثلاً وام خرید تجهیزات یا وام ودیعه مسکن) "
            
            "و مبلغ آن را بفرمایید.")

INCOMPLETE_LOAN = "متأسفانه نوع یا مبلغ وام مشخص نشد؛ لطفاً هر دو مورد را بفرمایید."


# ---- Angry customer (exact text from the project brief) + routing reason ----
ANGRY_MESSAGE = "مشتری گرامی، از مورد پیش آمده پوزش می‌طلبیم. درخواست شما مستقیماً به مدیر شعبه ارجاع داده شد."


ROUTE_TO_MANAGER = ("🔀 ارجاع به: مدیر شعبه (اپراتور انسانی) | دلیل: نارضایتی شما از روند رسیدگی تشخیص داده شد؛ "
                    "چنین مواردی برای پیگیری ویژه و سریع‌تر، مستقیماً توسط مدیر شعبه بررسی می‌شوند.")


# ---- Request that is not about a loan ----
ROUTE_TO_CUSTOMER_SERVICE = ("🔀 ارجاع به: کارشناس امور مشتریان (اپراتور انسانی) | دلیل: این سامانه مخصوص بررسی "
                             "درخواست‌های تسهیلات است و درخواست شما خارج از حوزه کمیته اعتباری است؛ "
                             "همکاران ما در امور مشتریان به‌زودی پیگیری خواهند کرد.")



# ---- Documents complete -> sent to the credit committee ----
ROUTE_TO_COMMITTEE = (
    "مدارک اولیه شما کامل است ✅\n"
    "• کد ملی: {customer_id}\n"
    "• درخواست: {loan_request}\n"
    "🔀 ارجاع به: «عامل ریسک و کشف تقلب» و «عامل تطبیق مقررات»\n"
    "دلیل: عامل ریسک، سوابق بانکی شما (چک برگشتی، بدهی‌های جاری و لیست سیاه) را استعلام می‌کند و "
    "عامل مقررات، انطباق درخواست شما را با آخرین بخشنامه‌های بانک مرکزی بررسی می‌کند. "
    "سپس مدیر کمیته اعتباری تصمیم نهایی را اعلام خواهد کرد."
)



def agent_says(text: str) -> dict:
    # A message from the Triage agent, in the format of our `messages` log
    return {"agent": TRIAGE_AGENT, "content": text}

def customer_says(text: str) -> dict:
    # A message from the customer, in the format of our `messages` log
    return {"agent": CUSTOMER, "content": text}

print("✅ Agent messages ready.")

✅ Agent messages ready.


In [48]:
# =====================================================================
#  1) SENTIMENT: one focused job -> Normal or Angry
# =====================================================================
class SentimentAnalysis(BaseModel):
    sentiment: Literal["Normal", "Angry"] = Field(
        description="'Angry' if the customer shows anger, frustration, impatience, complaint or dissatisfaction or only wants to speak to the manager. "
                    "'Normal' if the message is polite or neutral.")
    reason: str = Field(description="One short sentence in ENGLISH explaining the decision. Never empty.")


SENTIMENT_SYSTEM_PROMPT = (
    "You are the sentiment classifier of a bank's front desk. Classify the tone of the customer's Persian message.\n"
    "Angry = the customer is angry, frustrated, impatient, complaining or dissatisfied with the bank or its service "
    "(for example complaining about delays, long waiting, bad service, or asking angrily 'why...?').\n"
    "Normal = a polite or neutral message, even if it asks for a loan or asks a question.\n\n"
    "Examples:\n"
    "'سلام، برای خرید خودرو ۱۰۰ میلیون تومان وام میخواستم.' -> Normal\n"
    "'ببخشید، شرایط وام ازدواج چیه؟' -> Normal\n"
    "'سه هفته است منتظرم و هیچکس جواب نمیده! واقعا از این بانک ناامید شدم.' -> Angry\n"
    "'چندین روز میگذره ولی هیچ کس به درخواستم رسیدگی نمیکنه!' -> Angry\n"
    "'این چه وضعشه؟ چرا هر بار میام میگید فردا بیا؟' -> Angry\n\n"
    "Write the reason in English."
)

In [49]:
# =====================================================================
#  2) REQUEST: intent + national ID + loan type + loan amount
# =====================================================================
# Description shared by both loan models: the loan type must be copied in PERSIAN from the customer's words

LOAN_TYPE_DESCRIPTION = (
    "The requested loan type, written in PERSIAN script (never English), copied from the customer's own words, "
    "e.g.  ,'وام خرید تجهیزات', 'وام ودیعه مسکن', 'وام خرید خودرو'. '' if not given.")


class RequestAnalysis(BaseModel):
    intent: Literal["loan_request", "other"] = Field(
        description="'loan_request' ONLY if the message is about a loan / facility (وام، تسهیلات). "
                    "Everything else is 'other', e.g. lost or blocked bank card, money transfer, "
                    "account balance, opening an account, cheque book.")
    
    national_id: str = Field(
        description="The 10-digit national ID written in the message, using English digits. '' if not given.")
    
    loan_type: str = Field(description=LOAN_TYPE_DESCRIPTION)
    
    loan_amount_toman: int = Field(
        description="Requested loan amount in Toman as an integer. 0 if not given.")



In [50]:
# =====================================================================
#  3) LOAN: the customer's answer to our question about the loan
# =====================================================================
class LoanInfo(BaseModel):
    loan_type: str = Field(description=LOAN_TYPE_DESCRIPTION)
    loan_amount_toman: int = Field(description="Requested loan amount in Toman as an integer. 0 if not given.")


# ---------------- Instructions (system prompts) ----------------
AMOUNT_RULES = (
    "Convert the amount to Toman as an integer. Examples: "
    "'۵۰۰ میلیون تومان' -> 500000000, '۲۰۰ میلیون' -> 200000000, '۲ میلیارد تومان' -> 2000000000, "
    "'۸۰ میلیون تومان' -> 80000000."
    "convert Rial to Toman"
)

LOAN_TYPE_RULES = (
    "Write loan_type in Persian, using the customer's own words. Example: for "
    "'۵۰۰ میلیون تومان وام خرید تجهیزات میخوام' -> loan_type = 'وام خرید تجهیزات'."
)

REQUEST_SYSTEM_PROMPT = (
    "You are the front desk of a bank's credit committee. "
    "Analyze the customer's message (written in Persian) and fill in all the fields. "
    "intent is 'loan_request' only when the customer talks about a loan (وام / تسهیلات); "
    "for example 'کارت بانکی من گم شده' (lost bank card) is 'other'. "
    "Persian digits must be converted to English digits. " + AMOUNT_RULES + " " + LOAN_TYPE_RULES +
    " Never invent information that is not written in the message: use '' or 0 for missing values."
)

LOAN_SYSTEM_PROMPT = (
    "You are the front desk of a bank. The customer answered the question about the requested loan "
    "(in Persian). Extract the loan type and the loan amount. " + AMOUNT_RULES + " " + LOAN_TYPE_RULES +
    " Never invent information: use '' or 0 for missing values."
)

In [51]:

# ---------------- LLMs that must answer in our structure ----------------
sentiment_analyzer = llm.with_structured_output(SentimentAnalysis)
request_analyzer   = llm.with_structured_output(RequestAnalysis)
loan_extractor     = llm.with_structured_output(LoanInfo)


def format_loan_request(loan_type: str, amount_toman: int) -> str:
    # Builds the `loan_request` text (type + amount). Returns '' if one of them is missing.
    if loan_type.strip() and amount_toman > 0:
        return f"{loan_type.strip()} - {amount_toman:,} تومان"
    return ""

print("✅ Structured-output analyzers ready.")

✅ Structured-output analyzers ready.


### 3.1.4 The Triage agent node
This is the agent itself. Read it from top to bottom like a checklist: in every pass the agent finds the
**first unfinished job**, does it, returns the update and the graph comes back for the next job.

In [52]:
def triage_agent(state: BankingState) -> dict:
    """Node 1: Triage & Sentiment agent (front desk). Does ONE job per pass and returns the update."""

    # ---------- PASS 1: professional welcome ----------
    if not state["messages"]:
        user_text = state["user_input"]
        if not user_text:
            # no message yet -> show the welcome and WAIT for the customer's answer
            user_text = interrupt(WELCOME_MESSAGE)
        return {"user_input": user_text,
                "messages": [agent_says(WELCOME_MESSAGE), customer_says(user_text)]}

    

    # ---------- PASS 2: understand the customer's message (LLM, only once) ----------
    if not state["sentiment"]:
        # (1) Sentiment first: one focused LLM call
        mood = sentiment_analyzer.invoke([SystemMessage(content=SENTIMENT_SYSTEM_PROMPT),
                                          HumanMessage(content=state["user_input"])])
        print(f"   🧠 LLM sentiment: {mood}")


        # (a) Angry customer -> stop and refer to the branch manager (human operator)
        if mood.sentiment == "Angry":
            return {"sentiment": "Angry", "route": "human_operator",
                    "messages": [agent_says(ANGRY_MESSAGE), agent_says(ROUTE_TO_MANAGER)]}



        # (2) Tone is Normal -> second LLM call: what does the customer want, which documents were given?
        analysis = request_analyzer.invoke([SystemMessage(content=REQUEST_SYSTEM_PROMPT),
                                            HumanMessage(content=state["user_input"])])
        print(f"   🧠 LLM request analysis: {analysis}")



        # (b) Not a loan request -> refer to customer service (human operator)
        if analysis.intent == "other":
            return {"sentiment": "Normal", "route": "human_operator",
                    "messages": [agent_says(ROUTE_TO_CUSTOMER_SERVICE)]}



        # (c) Normal loan request -> store whatever information the customer already gave
        return {"sentiment": "Normal",
                "customer_id": normalize_national_id(analysis.national_id),   # '' if missing/invalid
                "loan_request": format_loan_request(analysis.loan_type, analysis.loan_amount_toman)}

    

    # ---------- PASS 3: national ID missing -> ask for it ----------
    if not state["customer_id"]:
        answer = interrupt(ASK_NATIONAL_ID)              # pause and wait for the customer's answer
        national_id = normalize_national_id(answer)       # Persian digits -> English, check 10 digits
        new_messages = [agent_says(ASK_NATIONAL_ID), customer_says(answer)]
        if not national_id:
            new_messages.append(agent_says(INVALID_NATIONAL_ID))   # invalid -> it will be asked again
        return {"customer_id": national_id, "messages": new_messages}



    # ---------- PASS 4: loan type / amount missing -> ask for it ----------
    if not state["loan_request"]:
        answer = interrupt(ASK_LOAN)                      # pause and wait for the customer's answer
        loan = loan_extractor.invoke([SystemMessage(content=LOAN_SYSTEM_PROMPT),
                                      HumanMessage(content=answer)])
        print(f"   🧠 LLM loan extraction: {loan}")
        loan_request = format_loan_request(loan.loan_type, loan.loan_amount_toman)
        new_messages = [agent_says(ASK_LOAN), customer_says(answer)]
        if not loan_request:
            new_messages.append(agent_says(INCOMPLETE_LOAN))      # incomplete -> it will be asked again
        return {"loan_request": loan_request, "messages": new_messages}
    

    # ---------- PASS 5: documents complete -> send the case to the credit committee ----------
    routing_message = ROUTE_TO_COMMITTEE.format(customer_id=state["customer_id"],
                                                loan_request=state["loan_request"])
    return {"documents_ok": True, "route": "credit_committee",
            "messages": [agent_says(routing_message)]}

### 3.1.5 The routing function (conditional edge)
After every pass, LangGraph calls this function to decide **where to go next**. It only reads the State and
returns the **name of the next path**. This is exactly the conditional rule of the project brief:
*if `sentiment == 'Angry'` → the graph ends (referral to a human).*

In [53]:
def route_after_triage(state: BankingState) -> str:
    # Decides where the graph goes after the Triage agent
    if state["sentiment"] == "Angry":
        return "human_operator"          # brief's rule: angry customer -> END, referral to a human
    if state["route"] == "human_operator":
        return "human_operator"          # not a loan request -> END, referral to customer service
    if state["documents_ok"]:
        return "credit_committee"        # everything complete -> Risk & Compliance agents
    return "triage"                      # still collecting information -> back to the Triage agent

### 3.1.7 Helper: running a conversation
This small helper talks to the graph for us:
1. `graph.invoke(...)` runs the graph until it **ends** or **pauses**.
2. If it paused, the result contains `"__interrupt__"` with the agent's **question**.
3. We get the customer's **answer**: from a prepared list (for demos) or typed live with `input()`.
4. We **resume** the graph with `Command(resume=answer)` and repeat, until the graph ends.
5. Finally we print the complete case file (State).

In [54]:
def run_conversation(graph, user_input: str = "", answers: list = None) -> dict:
    # Runs one customer conversation with the graph.
    # answers = prepared customer answers (for demos). If empty/None, the customer types live.
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}   # a new case number for every conversation
    answers = list(answers) if answers else None                  # copy, so the original list is not changed
    graph_input = create_initial_state(user_input)

    while True:
        result = graph.invoke(graph_input, config)          # run until the graph ENDS or PAUSES

        if "__interrupt__" not in result:                    # no pause -> the conversation is over
            break

        question = result["__interrupt__"][0].value          # the graph paused and asked a question
        print(f"🤖 Agent asks: {question}")
        if answers:
            answer = answers.pop(0)                          # next prepared answer
            print(f"👤 Customer answers: {answer}")
        else:
            answer = input("👤 Your answer: ")              # live: the customer types the answer
        graph_input = Command(resume=answer)                # resume the graph with the answer

    print_state(result, "Final State (case file)")
    return result

---
## 3.2 🕵️ Risk & Fraud Agent (Tool Caller)

**Role:** the financial risk analyst of the committee.

**Job:** take the customer's national ID (`customer_id`, stored by the Triage agent) and run the bank inquiries
with the **tools of Step 2**: credit score, bounced cheques, current debts and blacklist.

**Output:** a numeric + text report in `risk_report`, with the **credit score** and the **red flags** 🚩.

### How the agent works (5 small steps)
```
customer_id ─► 1) LLM chooses the tools ─► 2) our code runs them ─► 3) safety net
            ─► 4) bank rules: red flags + risk level ─► 5) LLM writes a short summary ─► risk_report
```

| Step | Who does it | Why |
|---|---|---|
| 1. Choose the tools | 🤖 LLM (`bind_tools`) | This is the **Tool Calling** part of the project |
| 2. Run the tools | 🐍 Our code | The LLM only *asks*; our code does the real call |
| 3. Safety net | 🐍 Our code | If the LLM forgets a tool, we call it ourselves, so the report is **always complete** |
| 4. Red flags & risk level | 🐍 Fixed bank rules | Risk decisions must be **clear and repeatable**, not guessed by an LLM |
| 5. Summary text | 🤖 LLM | The LLM is good at turning numbers into a short readable explanation |

### 3.2.1 The bank's risk rules
These are the simple, transparent rules our analyst uses to raise **red flags** 🚩:

| Rule | Red flag when… |
|---|---|
| Blacklist | the customer is on the bank's blacklist |
| Bounced cheques | the customer has **at least 1** bounced cheque |
| Overdue installments | the customer has **at least 1** overdue installment |
| Credit score | the score is **below 600** |
| Current debt | total debt is **more than 500,000,000 Toman** |

And the **risk level** is decided from the red flags:

| Situation | Risk level |
|---|---|
| Customer is blacklisted | ⛔ `Critical` |
| No red flags | 🟢 `Low` |
| 1–2 red flags | 🟡 `Medium` |
| 3 or more red flags | 🔴 `High` |

In [55]:
# ---------------- The bank's risk rules (thresholds) ----------------
MIN_CREDIT_SCORE = 600              # below this score -> red flag
HIGH_DEBT_LIMIT  = 500_000_000      # total debt above this (Toman) -> red flag


def find_red_flags(data: dict) -> list:
    # Applies the bank's rules to the inquiry results and returns the list of red flags
    red_flags = []
    if data["blacklisted"]:
        red_flags.append(f"Customer is on the bank blacklist ({data['blacklist_reason']})")
    if data["bounced_checks"] > 0:
        red_flags.append(f"{data['bounced_checks']} bounced cheque(s), total {data['bounced_checks_amount']:,} Toman")
    if data["overdue_installments"] > 0:
        red_flags.append(f"{data['overdue_installments']} overdue loan installment(s)")
    if data["credit_score"] < MIN_CREDIT_SCORE:
        red_flags.append(f"Low credit score ({data['credit_score']} < {MIN_CREDIT_SCORE})")
    if data["total_debt"] > HIGH_DEBT_LIMIT:
        red_flags.append(f"High current debt ({data['total_debt']:,} Toman)")
    return red_flags


def decide_risk_level(blacklisted: bool, red_flags: list) -> str:
    # Turns the red flags into one risk level
    if blacklisted:
        return "Critical"
    if len(red_flags) == 0:
        return "Low"
    if len(red_flags) <= 2:
        return "Medium"
    return "High"

print("✅ Risk rules ready.")

✅ Risk rules ready.


### 3.2.2 The Risk & Fraud agent node
Read the function from top to bottom: it follows exactly the 5 steps of the table above.

At the end, the agent returns only the fields it changed:
- `risk_report` → the complete report (numbers + red flags + risk level + summary)
- `messages` → a short note to the customer that the bank inquiries are done

In [56]:
RISK_AGENT = "Risk & Fraud Agent"

# The LLM of this agent can see and call the 4 banking tools
risk_llm = llm.bind_tools(banking_tools)

RISK_SYSTEM_PROMPT = (
    "You are the Risk & Fraud analyst of a bank's credit committee. "
    "For the customer's national ID, call ALL the available banking tools: "
    "credit score, bounced cheques, current debts and blacklist."
)

SUMMARY_SYSTEM_PROMPT = (
    "You are a bank risk analyst. You receive a customer's risk report as JSON. "
    "Write a short summary in English (2-3 sentences) for the head of the credit committee: "
    "mention the credit score, the risk level and every red flag. "
    "Do NOT approve or reject the loan; that is the committee head's job."
)

RISK_DONE_MESSAGE = ("استعلام‌های بانکی شما (نمره اعتباری، چک‌های برگشتی، بدهی‌های جاری و لیست سیاه) انجام شد "
                     "و گزارش ریسک برای مدیر کمیته اعتباری ارسال شد.")


def risk_fraud_agent(state: BankingState) -> dict:
    """Node 2: Risk & Fraud agent. Runs the bank inquiries with tools and builds the risk report."""
    national_id = state["customer_id"]

    # ---------- 1) The LLM chooses the tools ----------
    ai_message = risk_llm.invoke([
        SystemMessage(content=RISK_SYSTEM_PROMPT),
        HumanMessage(content=f"Customer national ID: {national_id}. Loan request: {state['loan_request']}"),
    ])

    # ---------- 2) Our code runs the tools the LLM asked for ----------
    results = {}                                        # tool name -> tool result
    for call in ai_message.tool_calls:
        results[call["name"]] = tools_by_name[call["name"]].invoke(call["args"])
        print(f"   🔧 LLM called: {call['name']}({call['args']})")

    # ---------- 3) Safety net: call any tool the LLM forgot ----------
    for banking_tool in banking_tools:
        if banking_tool.name not in results:
            results[banking_tool.name] = banking_tool.invoke({"national_id": national_id})
            print(f"   🛟 Safety net: the LLM did not call {banking_tool.name}, so we called it directly")

    # ---------- 4) Collect the numbers and apply the bank rules ----------
    score   = results["get_credit_score"]
    bounced = results["check_bounced_checks"]
    debts   = results["check_current_debts"]
    black   = results["check_blacklist"]

    risk_report = {
        "national_id":           national_id,
        "credit_score":          score["credit_score"],
        "credit_rating":         score["rating"],
        "bounced_checks":        bounced["bounced_checks"],
        "bounced_checks_amount": bounced["bounced_checks_amount"],
        "active_loans":          debts["active_loans"],
        "total_debt":            debts["total_debt"],
        "overdue_installments":  debts["overdue_installments"],
        "blacklisted":           black["blacklisted"],
        "blacklist_reason":      black["reason"],
    }
    risk_report["red_flags"]  = find_red_flags(risk_report)
    risk_report["risk_level"] = decide_risk_level(risk_report["blacklisted"], risk_report["red_flags"])

    # ---------- 5) The LLM writes a short summary of the report ----------
    summary = llm.invoke([
        SystemMessage(content=SUMMARY_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(risk_report, ensure_ascii=False)),
    ])
    risk_report["summary"] = summary.content

    return {"risk_report": risk_report,
            "messages": [{"agent": RISK_AGENT, "content": RISK_DONE_MESSAGE}]}

### 3.2.3 Testing the agent
A node is just a Python function, so we can test it **directly**: we give it a case file (State) and look at
what it returns. We test our 3 test customers.

This helper creates a case file as the Triage agent would leave it (national ID + loan request already stored):

In [57]:
def make_case(national_id: str, loan_request: str) -> BankingState:
    # A case file as the Triage agent leaves it: documents complete, tone normal
    case = create_initial_state("(test)")
    case["customer_id"]  = national_id
    case["loan_request"] = loan_request
    case["sentiment"]    = "Normal"
    case["documents_ok"] = True
    return case


def show_risk_result(update: dict) -> None:
    # Prints what the Risk & Fraud agent returned
    report = update["risk_report"]
    print(f"\n📊 Credit score : {report['credit_score']} ({report['credit_rating']})")
    print(f"⚠️ Risk level   : {report['risk_level']}")
    print("🚩 Red flags    :", report["red_flags"] if report["red_flags"] else "None")
    print(f"📝 Summary      : {report['summary']}")
    print(f"💬 Message      : [{update['messages'][0]['agent']}] {update['messages'][0]['content']}")

#### Test A — Scenario 1 customer (good customer)
We continue **the same case file** of Scenario 1 that the Triage agent created in Test A of Step 3.1 (`state_A`).

**👀 What to look for:** credit score **800**, **no** bounced cheques, **no** red flags → risk level `Low`.

In [58]:
# Run the Risk & Fraud agent on the Scenario 1 case file and show the result
risk_update_A = risk_fraud_agent(state_A)
show_risk_result(risk_update_A)

NameError: name 'state_A' is not defined

#### Test B — Risky customer
**👀 What to look for:** low score, bounced cheques, overdue installments and high debt → several red flags → `High`.

In [ ]:
# A case file for the risky customer, then run the agent
case_risky = make_case("0987654321", "وام ودیعه مسکن - 200,000,000 تومان")
risk_update_B = risk_fraud_agent(case_risky)
show_risk_result(risk_update_B)

#### Test C — Blacklisted customer (fraud)
**👀 What to look for:** the blacklist red flag → risk level `Critical`.

In [ ]:
# A case file for the blacklisted customer, then run the agent
case_fraud = make_case("1111111111", "وام خرید خودرو - 300,000,000 تومان")
risk_update_C = risk_fraud_agent(case_fraud)
show_risk_result(risk_update_C)

#### The complete `risk_report` saved in the State (Scenario 1)
This is the full dictionary that goes into the `risk_report` field of the State. In Step 3.4 the
**Supervisor / Judge** will read it to make the final decision.

In [ ]:
# Print the whole risk_report dictionary (ensure_ascii=False -> Persian text is readable)
print(json.dumps(risk_update_A["risk_report"], ensure_ascii=False, indent=4))

---
### 📝 Step 3.2 — Summary
| What we did | Why |
|---|---|
| `llm.bind_tools(banking_tools)` → the LLM chooses the tools | Tool Calling, as required by the brief |
| Our code runs the tools the LLM asked for | The LLM never runs code itself |
| Safety net for forgotten tools | The report is always complete, even if the LLM makes a mistake |
| Fixed bank rules for red flags 🚩 and risk level | Clear, explainable and repeatable risk decisions |
| LLM writes a short summary | Easy for the committee head to read |
| Returned `risk_report` + one message | The node only returns the fields it changed |

➡️ Next, in **Step 3.3**, we build the **Compliance agent** with **RAG**: we load the Central Bank circulars into a
**Vectorstore** and check if the loan request follows the rules.

---
## 3.3 ⚖️ Compliance Agent (RAG)

**Role:** the legal expert of the committee.

**Job:** check the requested loan type against the **latest Central Bank circulars** (بخشنامه‌ها), which are
stored as **PDF files** in a **Vectorstore**.

**Output:** `compliance_check` → is the request legal? Which collateral is needed?
(e.g. *"according to the new circular, a housing deposit loan needs two official guarantors"*)

### What is RAG (Retrieval-Augmented Generation)?
The LLM does **not** know our bank's circulars, and they can change every month. Instead of retraining the
model, we give it the **right pages** at the moment of the question:

```
loan_request ─► 1) RETRIEVE: search the Vectorstore ─► the related circulars
             ─► 2) AUGMENT: put the circulars into the prompt
             ─► 3) GENERATE: the LLM answers using ONLY those circulars ─► compliance_check
```

### Our circulars 📄
We wrote **4 sample circulars in Persian** and saved them as PDFs in the `circulars/` folder.
They are **fictional** and made only for this project.

| File | Circular | Main rule |
|---|---|---|
| `circular_01_equipment_loan.pdf` | 1405/101 – Equipment loan | max 1 billion Toman; **promissory note + business license** |
| `circular_02_housing_deposit_loan.pdf` | 1405/102 – Housing deposit loan | max 300 million Toman; **two official guarantors** |
| `circular_03_car_loan.pdf` | 1405/103 – Car loan | max 400 million Toman; car document + one guarantor |
| `circular_04_general_rules.pdf` | 1405/100 – General rules | no loans for bounced cheques / blacklist; amount above max is not allowed |

### 3.3.1 Load the PDF circulars
`PyPDFDirectoryLoader` reads **every PDF** in a folder and turns each page into a `Document`
(the text + metadata such as the file name).

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

circulars = PyPDFDirectoryLoader("circulars").load()     # read all PDFs in the folder

print(f"Loaded {len(circulars)} circulars:")
for doc in circulars:
    print("  📄", doc.metadata["source"])

print("\n--- Text of the first circular ---")
print(circulars[0].page_content)

> 📝 Persian text taken out of a PDF sometimes has small spacing errors (e.g. «اعتبار ی» instead of «اعتباری»).
> This does not matter: the embedding model and the LLM still understand the text.

### 3.3.2 Split the text into chunks
Real circulars can be many pages long, so we cut them into smaller **chunks** before searching.
Our circulars are short, so each one stays almost in one piece. That's good, because the **loan type stays
together with its rules**.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cut the circulars into chunks of max 800 characters (100 characters overlap between chunks)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(circulars)

print(f"{len(circulars)} circulars -> {len(chunks)} chunks")

### 3.3.3 Build the Vectorstore
1. The **embedding model** turns every chunk into a **vector** (a list of numbers that represents its meaning).
2. **Chroma** stores these vectors. Later it finds the chunks whose meaning is **closest** to a question.

> 💡 **Lesson learned: choosing the embedding model.** We first used `nomic-embed-text`, but it is trained
> mostly on **English**. For the request *"وام خرید تجهیزات"* it returned the **car loan** circular first ❌.
> We switched to `bge-m3`, a **multilingual** model that understands Persian much better ✅
> (set in Step 0 as `EMBEDDING_MODEL`).

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)       # bge-m3 (multilingual)

vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    collection_name="central_bank_circulars",
    ids=[f"chunk-{i}" for i in range(len(chunks))],        # fixed ids -> no duplicates if we run the cell again
)

# The retriever returns the 2 most similar chunks for a question
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Vectorstore ready with", len(chunks), "chunks.")

### 3.3.4 Test the search (Retrieval)
**👀 What to look for:** each loan type should find **its own circular** first.

In [ ]:
# Search the Vectorstore for each loan type and show which circular files were found
for question in ["وام خرید تجهیزات", "وام ودیعه مسکن", "وام خرید خودرو"]:
    found = retriever.invoke(question)
    print(f"🔎 {question}  ->  {[doc.metadata['source'] for doc in found]}")

### 3.3.5 The Compliance agent node
The agent does the 3 steps of RAG:
1. **Retrieve** the 2 circulars closest to `loan_request`.
2. **Augment**: put them into the message for the LLM.
3. **Generate**: the LLM answers in a **fixed 5-line Persian format**:

```
نتیجه: مجاز / غیرمجاز
بخشنامه: ...
سقف مبلغ: ...
وثایق لازم: ...
دلیل: ...
```

> 💡 **Lesson learned:** we first used *structured output* (a Pydantic model) like in Step 3.1. But when the
> answer had to contain **long Persian text**, our small 7B model sometimes dropped items (e.g. it forgot the
> business license) or even invented a fake link as the circular number. Because `compliance_check` is a
> **text** field anyway, we switched to a short **fixed text format**: the code became simpler, the answers
> became correct, and it runs about **2× faster**.

In [ ]:
COMPLIANCE_AGENT = "Compliance Agent"

COMPLIANCE_DONE_MESSAGE = ("بررسی انطباق درخواست شما با آخرین بخشنامه‌های بانک مرکزی انجام شد "
                           "و نتیجه برای مدیر کمیته اعتباری ارسال شد.")

COMPLIANCE_PROMPT = """You are the legal expert of a bank's credit committee.
Check the customer's loan request using ONLY the Central Bank circulars given to you.
- Is there a circular for this loan type?
- Is the requested amount within the maximum amount of that circular?
- Which collateral and documents does the circular require? Copy them exactly from the circular.

Answer in Persian, in exactly these 5 lines and nothing else:
نتیجه: مجاز یا غیرمجاز
بخشنامه: شماره بخشنامه
سقف مبلغ: سقف مبلغ به تومان
وثایق لازم: وثایق و مدارک لازم طبق بخشنامه
دلیل: یک جمله کوتاه"""


def compliance_agent(state: BankingState) -> dict:
    """Node 3: Compliance agent. Checks the loan request against the circulars (RAG)."""
    # 1) RETRIEVE: find the circulars related to the loan request
    found_docs = retriever.invoke(state["loan_request"])
    circulars_text = "\n\n".join(doc.page_content for doc in found_docs)

    # 2) AUGMENT: put the circulars and the request together
    question = f"Circulars:\n{circulars_text}\n\nLoan request: {state['loan_request']}"

    # 3) GENERATE: the LLM answers using only these circulars
    answer = llm.invoke([SystemMessage(content=COMPLIANCE_PROMPT), HumanMessage(content=question)])

    return {"compliance_check": answer.content,
            "messages": [{"agent": COMPLIANCE_AGENT, "content": COMPLIANCE_DONE_MESSAGE}]}

### 3.3.6 Testing the agent
Like in Step 3.2, we call the node directly with a case file.

#### Test A — Scenario 1: equipment loan, 500 million Toman
**👀 What to look for:** `مجاز`, circular **1405/101**, collateral = **سفته + پروانه کسب معتبر**
(exactly what the brief expects: *"نیاز به سفته و پروانه کسب"*).

In [ ]:
compliance_A = compliance_agent(state_A)            # the same Scenario 1 case file as before
print(compliance_A["compliance_check"])

#### Test B — Housing deposit loan, 200 million Toman
**👀 What to look for:** `مجاز`, and the new rule of circular 1405/102: **دو ضامن رسمی** (two official guarantors,
the example of the brief).

In [ ]:
# Housing deposit loan of 200 million Toman
compliance_B = compliance_agent(make_case("0987654321", "وام ودیعه مسکن - 200,000,000 تومان"))
print(compliance_B["compliance_check"])

#### Test C — Car loan, 600 million Toman (above the maximum!)
**👀 What to look for:** `غیرمجاز`, because the maximum for a car loan is **400 million Toman**.

In [ ]:
# Car loan of 600 million Toman (the maximum is 400 million)
compliance_C = compliance_agent(make_case("1234567890", "وام خرید خودرو - 600,000,000 تومان"))
print(compliance_C["compliance_check"])

#### Test D — A loan type without any circular (travel loan)
**👀 What to look for:** `غیرمجاز`, because no circular exists for this loan type.

In [ ]:
# A loan type that has no circular
compliance_D = compliance_agent(make_case("1234567890", "وام سفر خارجی - 100,000,000 تومان"))
print(compliance_D["compliance_check"])

---
### 📝 Step 3.3 — Summary
| What we did | Why |
|---|---|
| Wrote 4 Persian sample circulars as PDFs | The knowledge the LLM does not have |
| `PyPDFDirectoryLoader` + `RecursiveCharacterTextSplitter` | Read the PDFs and cut them into chunks |
| `bge-m3` embeddings + **Chroma** Vectorstore | Find the right circular by meaning (works well in Persian) |
| Retrieve → Augment → Generate | The LLM answers **only** from the bank's circulars, not from its memory |
| A fixed 5-line Persian answer | Simple, fast and correct with a small local model |

➡️ Next, in **Step 3.4**, we build the **Supervisor / Judge**, who reads the risk report and the compliance check
and writes the final **JSON** decision.

---
## 3.4 👨‍⚖️ Supervisor / Judge (Head of the Credit Committee)

**Role:** the manager of the credit committee.

**Job:** read the two reports and make the **final decision**:
- 🕵️ `risk_report` from the Risk & Fraud agent (credit score, risk level, red flags)
- ⚖️ `compliance_check` from the Compliance agent (allowed or not, collateral)

**Output:** a **structured JSON** in `final_decision`, exactly like the brief:
```json
{"status": "Approved", "max_amount": 500000000, "collateral": "Promissory Note & Business License"}
```
We also add a short `reason`, so every decision can be explained. Finally, the judge announces the decision
to the customer in Persian.

> 🛟 **Safety rule:** in our tests the LLM always chose the right status, but for rejected loans it sometimes
> still filled in an amount. So after the LLM answers, our code makes sure a **rejected** loan always has
> `max_amount = 0` and `collateral = "None"`.

### The committee's decision rules
| Situation | Decision |
|---|---|
| Risk level is `High` or `Critical` | ❌ Rejected |
| Compliance check says `غیرمجاز` (not allowed) | ❌ Rejected |
| Otherwise | ✅ Approved, `max_amount` = the requested amount |

### 3.4.1 The structure of the final decision
We use **structured output** again (like in Step 3.1). Here it works well, because the fields are **short**:
a fixed status, a number and a few English words.

`decision.model_dump()` turns the Pydantic object into a normal Python **dictionary**, which is exactly the
JSON we store in `final_decision`.

In [ ]:
SUPERVISOR = "Supervisor / Judge"     # the name shown in the conversation log

# The structure of the final JSON decision
class FinalDecision(BaseModel):
    status: Literal["Approved", "Rejected"] = Field(description="The final decision of the committee")
    max_amount: int = Field(description="If Approved: the requested loan amount in Toman. If Rejected: 0")
    collateral: str = Field(description="If Approved: the required collateral in short English, "
                                        "e.g. 'Promissory Note & Business License'. If Rejected: 'None'")
    reason: str = Field(description="One short English sentence explaining the decision")

# The LLM must answer in the FinalDecision structure
judge_llm = llm.with_structured_output(FinalDecision)

# Instructions for the judge: the decision rules + how to write the collateral

JUDGE_PROMPT = (
    "You are the head of a bank's credit committee. Make the final decision about the loan request "
    "using the risk report and the compliance check.\n"
    "Decision rules:\n"
    "- Reject if the risk level is High or Critical.\n"
    "- Reject if the compliance check says غیرمجاز (not allowed).\n"
    "- Otherwise approve.\n"
    "Write the collateral in English with these names, joined with ' & ': "
    "سفته = Promissory Note, پروانه کسب = Business License, ضامن رسمی = Official Guarantor, "
    "سند خودرو = Car Document. Example: 'Promissory Note & Business License'."
)

### 3.4.2 The message to the customer
The decision is announced to the customer with a fixed, polite Persian text (like the other agents).
For an approved loan we show the amount and the collateral. The collateral is taken from the
`وثایق لازم:` line of the compliance check. Thanks to the fixed 5-line format of Step 3.3, this line is easy to find.

In [ ]:
APPROVED_MESSAGE = ("مشتری گرامی، درخواست وام شما توسط کمیته اعتباری تأیید شد ✅\n"
                    "• سقف مبلغ مجاز: {amount:,} تومان\n"
                    "• وثایق لازم: {collateral}\n"
                    "لطفاً برای تکمیل مدارک به شعبه مراجعه فرمایید.")

REJECTED_MESSAGE = ("مشتری گرامی، متأسفانه درخواست وام شما در این مرحله توسط کمیته اعتباری تأیید نشد.\n"
                    "کارشناسان شعبه برای توضیح دلایل و راه‌های پیش رو با شما تماس خواهند گرفت.")


def get_line(text: str, title: str) -> str:
    # Returns the value of one line of the compliance check, e.g. get_line(text, "وثایق لازم")
    for line in text.splitlines():
        if line.strip().startswith(title):
            return line.split(":", 1)[1].strip()
    return ""

### 3.4.3 The Supervisor / Judge node

In [ ]:
def supervisor_agent(state: BankingState) -> dict:
    """Node 4: Supervisor / Judge. Combines the two reports and writes the final JSON decision."""
    report = state["risk_report"]

    # 1) Put the two reports together in one short text for the LLM
    case_summary = (
        f"Loan request: {state['loan_request']}\n"
        f"Risk report: credit score {report['credit_score']}, risk level {report['risk_level']}, "
        f"red flags: {report['red_flags'] or 'none'}\n"
        f"Compliance check:\n{state['compliance_check']}"
    )

    # 2) The LLM makes the decision in our fixed structure
    decision = judge_llm.invoke([SystemMessage(content=JUDGE_PROMPT), HumanMessage(content=case_summary)])
    final_decision = decision.model_dump()          # Pydantic object -> dictionary (JSON)

    # Safety rule: a rejected loan has no amount and no collateral
    if decision.status == "Rejected":
        final_decision["max_amount"] = 0
        final_decision["collateral"] = "None"

    # 3) Announce the decision to the customer
    if decision.status == "Approved":
        collateral_fa = get_line(state["compliance_check"], "وثایق لازم")
        customer_message = APPROVED_MESSAGE.format(amount=decision.max_amount, collateral=collateral_fa)
    else:
        customer_message = REJECTED_MESSAGE

    return {"final_decision": final_decision,
            "messages": [{"agent": SUPERVISOR, "content": customer_message}]}

### 3.4.4 Testing the judge
To test the judge, we build **complete case files**: the case file + the risk report of Step 3.2 + the
compliance check of Step 3.3. (In Step 3.5 the graph will do this automatically.)

#### Test A — Scenario 1: good customer, equipment loan
**👀 What to look for:** exactly the JSON of the brief:
`{"status": "Approved", "max_amount": 500000000, "collateral": "Promissory Note & Business License"}`

In [ ]:
full_case_A = {**state_A,
               "risk_report": risk_update_A["risk_report"],          # from Step 3.2
               "compliance_check": compliance_A["compliance_check"]} # from Step 3.3

decision_A = supervisor_agent(full_case_A)
print(json.dumps(decision_A["final_decision"], ensure_ascii=False, indent=4))
print("\n💬", decision_A["messages"][0]["content"])

In [ ]:
# Risky customer: the case file + its risk report + its compliance check
full_case_B = {**case_risky,
               "risk_report": risk_update_B["risk_report"],
               "compliance_check": compliance_B["compliance_check"]}

decision_B = supervisor_agent(full_case_B)
print(json.dumps(decision_B["final_decision"], ensure_ascii=False, indent=4))
print("\n💬", decision_B["messages"][0]["content"])

In [ ]:
full_case_C = {**make_case("1234567890", "وام خرید خودرو - 600,000,000 تومان"),
               "risk_report": risk_update_A["risk_report"],          # the good customer's risk report
               "compliance_check": compliance_C["compliance_check"]}

decision_C = supervisor_agent(full_case_C)
print(json.dumps(decision_C["final_decision"], ensure_ascii=False, indent=4))
print("\n💬", decision_C["messages"][0]["content"])

---
## 3.5 🔀 Connecting Everything: the Credit Committee Graph

Now we connect our 4 agents into **one graph**, exactly as the project brief asks:

1. ✅ The request **starts at the Triage node**.
2. ✅ **Conditional edge:** if `sentiment == 'Angry'` → the graph **ends** (referral to a human).
   Otherwise the case goes to the **Risk** (Tools) and **Compliance** (RAG) agents, which run **in parallel**.
3. ✅ In the end, **all information goes to the Supervisor / Judge**.

```
                       ┌──── info missing: ask again ────┐
                       ▼                                 │
START ──► triage_agent ─┴─┬─ Angry / not a loan ──► END (human operator)
                          │
                          └─ documents complete ─┬─► risk_fraud_agent ──┐
                                                 │                      ├─► supervisor_agent ──► END
                                                 └─► compliance_agent ──┘
```

### Two important ideas
- **Parallel run (fan-out):** the Risk and Compliance agents don't need each other's results, so they can
  work **at the same time**. Both write a message into `messages`, and thanks to the **reducer** we defined in
  Step 1 (`operator.add`), no message is lost.
- **Waiting for both (fan-in):** the Supervisor must wait until **both** reports are ready. In LangGraph we
  write this as one edge from a **list** of nodes: `add_edge(["risk_fraud_agent", "compliance_agent"], "supervisor_agent")`.

### 3.5.1 The routing function after Triage
In Step 3.1, `route_after_triage()` returned one of 3 answers. Now we turn each answer into the real next step.
To start **two agents in parallel**, the routing function simply returns a **list** of node names.

In [ ]:
def route_from_triage(state: BankingState):
    # Decides where the case goes after the Triage agent (uses the rules of Step 3.1)
    decision = route_after_triage(state)

    if decision == "human_operator":                    # Angry customer or not a loan request
        return END                                      # -> the graph stops (referral to a human)
    if decision == "credit_committee":                  # documents complete
        return ["risk_fraud_agent", "compliance_agent"] # -> both agents start in PARALLEL
    return "triage_agent"                               # information missing -> ask again

### 3.5.2 Building the graph

In [ ]:
builder = StateGraph(BankingState)

# 1) The 4 agents (nodes)
builder.add_node("triage_agent", triage_agent)            # Node 1: Triage & Sentiment
builder.add_node("risk_fraud_agent", risk_fraud_agent)    # Node 2: Risk & Fraud (Tools)
builder.add_node("compliance_agent", compliance_agent)    # Node 3: Compliance (RAG)
builder.add_node("supervisor_agent", supervisor_agent)    # Node 4: Supervisor / Judge

# 2) Every request starts at the Triage agent
builder.add_edge(START, "triage_agent")

# 3) Conditional edge: Angry -> END, complete -> Risk + Compliance, missing info -> Triage again
builder.add_conditional_edges("triage_agent", route_from_triage,
                              ["triage_agent", "risk_fraud_agent", "compliance_agent", END])

# 4) The Supervisor waits for BOTH reports, then the graph ends
builder.add_edge(["risk_fraud_agent", "compliance_agent"], "supervisor_agent")
builder.add_edge("supervisor_agent", END)

# 5) Compile with memory (needed so the Triage agent can pause and ask questions)
credit_committee_graph = builder.compile(checkpointer=InMemorySaver())

print("✅ Credit committee graph is ready!")
print("Nodes:", [name for name in credit_committee_graph.get_graph().nodes])

In [ ]:
from IPython.display import Image, display     # to show a picture inside the notebook

# Draw the compiled graph as a PNG picture and show it
display(Image(credit_committee_graph.get_graph().draw_mermaid_png()))

## 4.2 Helper: running a scenario and printing the State at every step
This is the same idea as `run_conversation()` in Step 3.1, but now we use `graph.stream(...)` so we can see
**every step**:
- `stream_mode="updates"` tells us **which agent** just ran.
- `stream_mode="values"` gives us the **complete State** after that step → we print it with `print_state()`.

If the graph pauses to ask the customer something (`__interrupt__`), we answer and resume, just like before.

> 💡 When the Risk and Compliance agents run **in parallel**, they finish in the **same step**, so you will see
> them together in one step: *"after: risk_fraud_agent + compliance_agent"*.

In [ ]:
def run_scenario(user_input: str = "", answers: list = None) -> dict:
    # Runs one customer through the whole committee and prints the State after every step
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}   # new case number
    answers = list(answers) if answers else None
    graph_input = create_initial_state(user_input)
    print_state(graph_input, "Step 0 – the new case file")
    step = 0

    while True:
        question, agents_that_ran = None, []
        for mode, chunk in credit_committee_graph.stream(graph_input, config, stream_mode=["updates", "values"]):
            if mode == "updates":                          # which agent(s) just finished
                if "__interrupt__" in chunk:
                    question = chunk["__interrupt__"][0].value
                else:
                    agents_that_ran += list(chunk)
            elif agents_that_ran:                          # "values": the full State after that step
                step += 1
                print_state(chunk, f"Step {step} – after: {' + '.join(agents_that_ran)}")
                agents_that_ran = []

        if question is None:                               # the graph reached END
            break
        print(f"\n🤖 Agent asks: {question}")              # the graph paused: answer and resume
        answer = answers.pop(0) if answers else input("👤 Your answer: ")
        print(f"👤 Customer answers: {answer}")
        graph_input = Command(resume=answer)

    return credit_committee_graph.get_state(config).values

## 4.3 🟢 Scenario 1 — Good customer, loan approved
**Input:** *«من برای توسعه کسب‌وکارم ۵۰۰ میلیون تومان وام خرید تجهیزات میخوام. کد ملی: ۱۲۳۴۵۶۷۸۹۰»*

**Expected flow (from the project brief):**
1. The tone is detected as **Normal**.
2. The tools report an **excellent credit score (800)** and **no bounced cheques**.
3. The RAG system finds the rules of the equipment loan: **promissory note + business license**.
4. The final JSON: `{"status": "Approved", "max_amount": 500000000, "collateral": "Promissory Note & Business License"}`

In [ ]:
# Run Scenario 1 through the whole committee (the State is printed after every step)
scenario_1_result = run_scenario(scenario_1_input)

### ✅ Scenario 1 — the final result

In [ ]:
# The final JSON decision and the last message (from the Supervisor) to the customer
print("📄 Final decision (JSON):")
print(json.dumps(scenario_1_result["final_decision"], ensure_ascii=False, indent=4))
print("\n💬 What the customer receives:")
print(scenario_1_result["messages"][-1]["content"])

## 4.4 🔴 Scenario 2 — Angry customer, the graph stops
**Input:** *«چرا وام من رو نمیدید؟ من یک ماهه تو صف هستم و واقعا از سرویس‌دهی بانک خسته شدم!»*

**Expected flow (from the project brief):** the Triage agent detects the tone as **Angry**, the graph **stops
immediately**, and the customer receives:
*«مشتری گرامی، از تاخیر پیش آمده پوزش می‌طلبیم. درخواست شما مستقیماً به مدیر شعبه ارجاع داده شد.»*

In [ ]:
# Run Scenario 2: the graph should stop right after the Triage agent
scenario_2_result = run_scenario(scenario_2_input)

### ✅ Scenario 2 — the final result

In [ ]:
# Show that the graph stopped: no risk report and no final decision were created
print("😠 Sentiment     :", scenario_2_result["sentiment"])
print("🔀 Route         :", scenario_2_result["route"])
print("🕵️ Risk report   :", scenario_2_result["risk_report"] or "not created (the graph stopped)")
print("📄 Final decision:", scenario_2_result["final_decision"] or "not created (the graph stopped)")
print("\n💬 What the customer receives:")
print(scenario_2_result["messages"][-2]["content"])   # the apology message of the brief

## 4.5 🎤 Live demo (optional)
Run this cell and be the customer yourself: the agent welcomes you, asks for your national ID and your loan,
and the whole committee decides. Try the national IDs of our test customers: `1234567890` (good),
`0987654321` (risky) or `1111111111` (blacklisted).

> ⚠️ This cell waits for your typing in the input box.

In [ ]:
# Be the customer yourself: type your answers in the input box
live_result = run_scenario()